# Day 16 — Gradient boosting: XGBoost and LightGBM

Run each cell with **Shift + Enter**, from the top. XGBoost and LightGBM come preinstalled in Colab; locally, `pip install xgboost lightgbm`. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Introduction to gradient boosting

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-016/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# Gradient boosting by hand: predict the exam SCORE from study hours, on the 127 training students (same split as Days 14-15)
df = pd.read_csv("student_data_150.csv")
train_val, test = train_test_split(df, test_size=0.15, random_state=42, stratify=df["Passed"])
X, y = train_val[["StudyHours"]], train_val["ExamScore"].to_numpy()
rate = 0.3                                            # learning rate (eta): how much of each tree's correction to keep

F = np.full(len(y), y.mean())                         # step 1: start with the average score for everyone
rmse = lambda pred: np.sqrt(np.mean((y - pred) ** 2))
print(f"Start: predict the average {y.mean():.1f} for everyone | training RMSE {rmse(F):.2f}")
for m in range(1, 4):
    residuals = y - F                                 # step 2: how wrong are we? (for squared error: the negative gradient)
    tree = DecisionTreeRegressor(max_depth=1).fit(X, residuals)       # step 3: a tiny tree learns the errors
    F = F + rate * tree.predict(X)                    # step 4: add a fraction of its correction
    t = tree.tree_
    print(f"Tree {m}: StudyHours <= {t.threshold[0]:.2f}? correct by {t.value[1][0][0]:+.1f} / {t.value[2][0][0]:+.1f} "
          f"(x {rate}) | training RMSE {rmse(F):.2f}")

Start: predict the average 57.3 for everyone | training RMSE 13.88
Tree 1: StudyHours <= 3.75? correct by -5.9 / +7.5 (x 0.3) | training RMSE 13.04
Tree 2: StudyHours <= 4.75? correct by -2.7 / +10.6 (x 0.3) | training RMSE 12.46
Tree 3: StudyHours <= 1.25? correct by -20.5 / +1.2 (x 0.3) | training RMSE 11.95


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Step 5, repeat: keep adding trees. Does scikit-learn's GradientBoostingRegressor do exactly this?
F, rows = np.full(len(y), y.mean()), []
for m in range(1, 101):
    F = F + rate * DecisionTreeRegressor(max_depth=1).fit(X, y - F).predict(X)
    if m in (1, 3, 10, 30, 100):
        rows.append({"Trees": m, "TrainingRMSE": round(rmse(F), 2)})
print(pd.DataFrame(rows).to_string(index=False))
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=rate, max_depth=1).fit(X, y)
print(f"Largest difference from scikit-learn's model: {np.abs(gbr.predict(X) - F).max():.6f} marks")

 Trees  TrainingRMSE
     1         13.04
     3         11.95
    10         10.79
    30         10.52
   100         10.51
Largest difference from scikit-learn's model: 0.000000 marks


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import recall_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Pass or fail: one tree vs bagging (random forest) vs boosting, 5-fold CV on the same 127 students
features = ["StudyHours", "Attendance", "PreviousScore"]
y_cls = train_val["Passed"].astype(int)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {"Decision tree (depth 5)": DecisionTreeClassifier(max_depth=5, random_state=42),
          "Random forest (100 trees)": RandomForestClassifier(n_estimators=100, random_state=42),
          "Gradient boosting (100 trees)": GradientBoostingClassifier(random_state=42)}
for name, m in models.items():
    acc = cross_val_score(m, train_val[features], y_cls, cv=cv5)
    fail_recall = recall_score(y_cls, cross_val_predict(m, train_val[features], y_cls, cv=cv5), pos_label=0)
    print(f"{name:<30} CV accuracy {acc.mean():.3f} +/- {acc.std():.3f} | recall for fail {fail_recall:.3f}")

Decision tree (depth 5)        CV accuracy 0.709 +/- 0.052 | recall for fail 0.486


Random forest (100 trees)      CV accuracy 0.804 +/- 0.063 | recall for fail 0.541


Gradient boosting (100 trees)  CV accuracy 0.742 +/- 0.076 | recall for fail 0.514


In [ ]:
import xgboost
import lightgbm
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Two optimized implementations, with their default settings, on the same folds
print(f"xgboost {xgboost.__version__} | lightgbm {lightgbm.__version__}")
for name, m in {"XGBoost": XGBClassifier(random_state=42),
                "LightGBM": LGBMClassifier(random_state=42, verbose=-1)}.items():
    acc = cross_val_score(m, train_val[features], y_cls, cv=cv5)
    print(f"{name:<9} CV accuracy {acc.mean():.3f} +/- {acc.std():.3f}")

xgboost 3.4.1 | lightgbm 4.7.0


XGBoost   CV accuracy 0.773 +/- 0.086


LightGBM  CV accuracy 0.780 +/- 0.056


## 2. How gradient boosting works

In [ ]:
from sklearn.metrics import mean_squared_error

# Section 2: residual learning on five students (A-E), as in the lesson
X5 = np.array([[1], [2], [3], [4], [5]])
y5 = np.array([50, 60, 70, 80, 90])
predictions = np.full(len(y5), np.mean(y5), dtype=float)        # F0 = the mean, 70
print(f"Start: every prediction {np.mean(y5):.0f} | residuals {(y5 - predictions).astype(int).tolist()} | MSE {mean_squared_error(y5, predictions):.2f}")
learning_rate = 0.1
for iteration in range(5):
    residuals = y5 - predictions                               # errors (for squared error: the negative gradient)
    tree = DecisionTreeRegressor(max_depth=1, random_state=42).fit(X5, residuals)
    corrections = tree.predict(X5)
    predictions += learning_rate * corrections                 # F_m = F_(m-1) + eta * h_m
    if iteration == 0:
        print(f"Tree 1 corrections {corrections.round(1).tolist()} -> x {learning_rate} -> predictions {predictions.round(2).tolist()}")
    print(f"Iteration {iteration + 1}: MSE = {mean_squared_error(y5, predictions):.2f}")

Start: every prediction 70 | residuals [-20, -10, 0, 10, 20] | MSE 200.00
Tree 1 corrections [-15.0, -15.0, 10.0, 10.0, 10.0] -> x 0.1 -> predictions [68.5, 68.5, 71.0, 71.0, 71.0]
Iteration 1: MSE = 171.50
Iteration 2: MSE = 146.67
Iteration 3: MSE = 126.56
Iteration 4: MSE = 109.15
Iteration 5: MSE = 95.02


In [ ]:
# The real implementation on the same five students (training data only: just a demonstration)
model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, loss="squared_error", random_state=42)
model.fit(X5, y5)
print("Actual:   ", y5.tolist())
print("Predicted:", model.predict(X5).round(2).tolist())
print(f"MSE: {mean_squared_error(y5, model.predict(X5)):.4f}")

Actual:    [50, 60, 70, 80, 90]
Predicted: [50.0, 60.0, 70.0, 80.0, 90.0]
MSE: 0.0000


In [ ]:
# Residuals vs negative gradients: what each loss asks the next tree to learn, for the five students at F0 = 70
r = y5 - 70
print("Residuals (actual - predicted):    ", r.tolist())
print("Squared error, negative gradient:  ", r.tolist(), " <- the residuals themselves (up to a factor 2)")
print("Absolute error, negative gradient: ", np.sign(r).astype(int).tolist(), " <- only the direction")
p = np.array([0.8, 0.6, 0.5, 0.4, 0.3]); passed = np.array([1, 1, 0, 0, 0])
print("Log loss, negative gradient y - p: ", (passed - p).round(1).tolist(), "for P(pass)", p.tolist())

Residuals (actual - predicted):     [-20, -10, 0, 10, 20]
Squared error, negative gradient:   [-20, -10, 0, 10, 20]  <- the residuals themselves (up to a factor 2)
Absolute error, negative gradient:  [-1, -1, 0, 1, 1]  <- only the direction
Log loss, negative gradient y - p:  [0.2, 0.4, -0.5, -0.4, -0.3] for P(pass) [0.8, 0.6, 0.5, 0.4, 0.3]


In [ ]:
from sklearn.model_selection import KFold, cross_validate

# Experiment: learning rate vs number of trees, predicting ExamScore from three features on the 127 (5-fold CV)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
Xs, ys = train_val[features], train_val["ExamScore"]
configs = [("A", 0.01, 100), ("B", 0.05, 100), ("C", 0.1, 100), ("D", 0.3, 100), ("E", 0.05, 300)]
for name, lr, n in configs:
    cv = cross_validate(GradientBoostingRegressor(learning_rate=lr, n_estimators=n, max_depth=3, random_state=42),
                        Xs, ys, cv=kf, scoring="neg_root_mean_squared_error", return_train_score=True)
    print(f"{name}: learning rate {lr:<4} trees {n:<3} | training RMSE {-cv['train_score'].mean():5.2f} | validation RMSE {-cv['test_score'].mean():5.2f}")
print(f"For scale: the scores spread by {ys.std(ddof=0):.2f} (the RMSE of predicting their average)")

A: learning rate 0.01 trees 100 | training RMSE  8.78 | validation RMSE 10.25


B: learning rate 0.05 trees 100 | training RMSE  5.34 | validation RMSE  9.90


C: learning rate 0.1  trees 100 | training RMSE  3.85 | validation RMSE 10.14


D: learning rate 0.3  trees 100 | training RMSE  1.13 | validation RMSE 10.84


E: learning rate 0.05 trees 300 | training RMSE  2.93 | validation RMSE 10.34
For scale: the scores spread by 13.88 (the RMSE of predicting their average)


In [ ]:
# Validation RMSE after every tree, for three learning rates (staged predictions, averaged over the 5 folds)
curves = {}
for lr in [0.01, 0.1, 0.3]:
    errs = np.zeros(300)
    for tr, va in kf.split(Xs):
        m = GradientBoostingRegressor(learning_rate=lr, n_estimators=300, max_depth=3, random_state=42).fit(Xs.iloc[tr], ys.iloc[tr])
        errs += [np.sqrt(mean_squared_error(ys.iloc[va], p)) for p in m.staged_predict(Xs.iloc[va])]
    curves[f"Rate{lr}"] = (errs / 5).round(3)
    best = int(np.argmin(curves[f"Rate{lr}"])) + 1
    print(f"learning rate {lr:<4}: best validation RMSE {curves[f'Rate{lr}'].min():.2f} at {best:3d} trees | after 300 trees {curves[f'Rate{lr}'][-1]:.2f}")
pd.DataFrame({"Trees": np.arange(1, 301), **curves}).iloc[::5].to_csv("learning_rate_curves.csv", index=False)

learning rate 0.01: best validation RMSE 9.79 at 235 trees | after 300 trees 9.84


learning rate 0.1 : best validation RMSE 9.79 at  21 trees | after 300 trees 10.88


learning rate 0.3 : best validation RMSE 9.59 at   7 trees | after 300 trees 11.14


## 3. XGBoost deep dive

In [ ]:
from sklearn.metrics import classification_report

# Section 3: XGBoost. The lesson's first model, judged out-of-fold on the 127 (1 = pass, 0 = fail; the test set waits)
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                    eval_metric="logloss", random_state=42)
pred = cross_val_predict(xgb, train_val[features], y_cls, cv=cv5)
print(classification_report(y_cls, pred, target_names=["Fail", "Pass"], digits=3).rstrip())

              precision    recall  f1-score   support

        Fail      0.655     0.514     0.576        37
        Pass      0.816     0.889     0.851        90

    accuracy                          0.780       127
   macro avg      0.736     0.701     0.713       127
weighted avg      0.769     0.780     0.771       127


In [ ]:
from sklearn.model_selection import cross_validate

# Regularization: reg_alpha (L1) and reg_lambda (L2), compared by 5-fold CV. Log loss: lower is better
for config in [{"reg_alpha": 0, "reg_lambda": 1}, {"reg_alpha": 0.1, "reg_lambda": 1}, {"reg_alpha": 1, "reg_lambda": 5},
               {"reg_alpha": 0, "reg_lambda": 50}]:
    m = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, eval_metric="logloss", random_state=42, **config)
    cv = cross_validate(m, train_val[features], y_cls, cv=cv5, scoring=["accuracy", "neg_log_loss"], return_train_score=True)
    print(f"{str(config):<36} train log loss {-cv['train_neg_log_loss'].mean():.3f} | CV log loss {-cv['test_neg_log_loss'].mean():.3f}"
          f" | CV accuracy {cv['test_accuracy'].mean():.3f}")

{'reg_alpha': 0, 'reg_lambda': 1}    train log loss 0.182 | CV log loss 0.497 | CV accuracy 0.757


{'reg_alpha': 0.1, 'reg_lambda': 1}  train log loss 0.188 | CV log loss 0.497 | CV accuracy 0.765


{'reg_alpha': 1, 'reg_lambda': 5}    train log loss 0.308 | CV log loss 0.461 | CV accuracy 0.796


{'reg_alpha': 0, 'reg_lambda': 50}   train log loss 0.415 | CV log loss 0.491 | CV accuracy 0.788


In [ ]:
# Learning rate with 100 trees: training vs validation log loss (5-fold CV)
for lr in [0.01, 0.05, 0.1, 0.3]:
    m = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=lr, eval_metric="logloss", random_state=42)
    cv = cross_validate(m, train_val[features], y_cls, cv=cv5, scoring=["accuracy", "neg_log_loss"], return_train_score=True)
    print(f"learning_rate {lr:<5} train log loss {-cv['train_neg_log_loss'].mean():.3f} | CV log loss {-cv['test_neg_log_loss'].mean():.3f}"
          f" | CV accuracy {cv['test_accuracy'].mean():.3f}")

learning_rate 0.01  train log loss 0.422 | CV log loss 0.503 | CV accuracy 0.773


learning_rate 0.05  train log loss 0.252 | CV log loss 0.457 | CV accuracy 0.780


learning_rate 0.1   train log loss 0.182 | CV log loss 0.497 | CV accuracy 0.757


learning_rate 0.3   train log loss 0.093 | CV log loss 0.603 | CV accuracy 0.765


In [ ]:
# Early stopping watches a VALIDATION set, never the test set. With few students one validation set is noisy,
# so run it once per CV fold: train on 4 folds, watch the 5th, stop 20 rounds after the last improvement
best, used, losses = [], [], []
for tr, va in cv5.split(train_val[features], y_cls):
    es = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=3, eval_metric="logloss",
                       early_stopping_rounds=20, random_state=42)
    es.fit(train_val[features].iloc[tr], y_cls.iloc[tr], eval_set=[(train_val[features].iloc[va], y_cls.iloc[va])], verbose=False)
    curve = es.evals_result()["validation_0"]["logloss"]
    best.append(es.best_iteration + 1); used.append(len(curve)); losses.append(curve[es.best_iteration])
print("Best number of trees per fold:", best)
print("Rounds trained per fold:      ", used, "(instead of 500)")
print("Best validation log loss:     ", np.round(losses, 3).tolist())
print(f"Median best: {int(np.median(best))} trees")

Best number of trees per fold: [49, 132, 101, 70, 62]
Rounds trained per fold:       [69, 152, 121, 90, 82] (instead of 500)
Best validation log loss:      [0.526, 0.493, 0.436, 0.42, 0.327]
Median best: 70 trees


In [ ]:
# Missing values: blank 20% of attendance at random. XGBoost learns where missing values should go at each split
holes = train_val.copy()
holes.loc[holes.sample(frac=0.2, random_state=1).index, "Attendance"] = np.nan
print(f"Missing attendance: {holes['Attendance'].isna().sum()} of {len(holes)}")
for name, data in [("complete data", train_val), ("20% attendance missing", holes)]:
    acc = cross_val_score(XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42),
                          data[features], y_cls, cv=cv5)
    print(f"XGBoost, {name:<24} CV accuracy {acc.mean():.3f}")
try:
    GradientBoostingClassifier(random_state=42).fit(holes[features], y_cls)
    print("scikit-learn GradientBoostingClassifier: trained with missing values too")
except ValueError as e:
    print("scikit-learn GradientBoostingClassifier:", str(e).split(".")[0])

Missing attendance: 25 of 127


XGBoost, complete data            CV accuracy 0.757


XGBoost, 20% attendance missing   CV accuracy 0.757
scikit-learn GradientBoostingClassifier: Input X contains NaN


## 4. LightGBM deep dive

In [ ]:
# Section 4: LightGBM. The lesson's first model, judged out-of-fold on the 127 (the test set waits for Section 5)
lgbm = LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=15, max_depth=5, min_child_samples=10,
                      random_state=42, verbosity=-1)
pred = cross_val_predict(lgbm, train_val[features], y_cls, cv=cv5)
print(classification_report(y_cls, pred, target_names=["Fail", "Pass"], digits=3).rstrip())

              precision    recall  f1-score   support

        Fail      0.692     0.486     0.571        37
        Pass      0.812     0.911     0.859        90

    accuracy                          0.787       127
   macro avg      0.752     0.699     0.715       127
weighted avg      0.777     0.787     0.775       127


In [ ]:
# Leaf-wise vs level-wise: grow ONE tree with 8 leaves each way and compare its shape
from xgboost import XGBRegressor
one_lgb = LGBMClassifier(n_estimators=1, num_leaves=8, max_depth=-1, min_child_samples=5, random_state=42, verbosity=-1)
one_lgb.fit(train_val[features], y_cls)
leaves = one_lgb.booster_.trees_to_dataframe().query("split_feature.isna()")
print(f"LightGBM (leaf-wise, num_leaves=8): {len(leaves)} leaves at depths {sorted(leaves['node_depth'].astype(int) - 1)}")
one_xgb = XGBClassifier(n_estimators=1, max_depth=3, min_child_weight=0, random_state=42).fit(train_val[features], y_cls)
t = one_xgb.get_booster().trees_to_dataframe()
depth = {row.ID: 0 for row in t.itertuples() if row.Node == 0}
for row in t.itertuples():
    for child in (row.Yes, row.No):
        if isinstance(child, str):
            depth[child] = depth[row.ID] + 1
xleaves = t[t["Feature"] == "Leaf"]
print(f"XGBoost  (level-wise, max_depth=3):  {len(xleaves)} leaves at depths {sorted(depth[i] for i in xleaves['ID'])}")

LightGBM (leaf-wise, num_leaves=8): 8 leaves at depths [2, 3, 3, 3, 3, 3, 4, 4]


XGBoost  (level-wise, max_depth=3):  8 leaves at depths [3, 3, 3, 3, 3, 3, 3, 3]


In [ ]:
# Histogram-based learning: values are grouped into bins (max_bin, default 255) before splits are searched
for f in features:
    print(f"{f:<14} {train_val[f].nunique():3d} distinct values -> with 255 bins, every value gets its own bin")
edges = np.histogram_bin_edges(train_val["StudyHours"], bins=4)
counts, _ = np.histogram(train_val["StudyHours"], bins=edges)
print("StudyHours in 4 bins:", [f"{a:.1f}-{b:.1f}: {c}" for a, b, c in zip(edges[:-1], edges[1:], counts)])
for max_bin in [255, 8, 4]:
    acc = cross_val_score(LGBMClassifier(max_bin=max_bin, random_state=42, verbosity=-1), train_val[features], y_cls, cv=cv5)
    print(f"max_bin={max_bin:<4} CV accuracy {acc.mean():.3f}")

StudyHours      14 distinct values -> with 255 bins, every value gets its own bin
Attendance      39 distinct values -> with 255 bins, every value gets its own bin
PreviousScore   56 distinct values -> with 255 bins, every value gets its own bin
StudyHours in 4 bins: ['0.5-2.1: 20', '2.1-3.8: 51', '3.8-5.4: 40', '5.4-7.0: 16']
max_bin=255  CV accuracy 0.780


max_bin=8    CV accuracy 0.780


max_bin=4    CV accuracy 0.827


In [ ]:
from sklearn.metrics import f1_score

# Experiment: num_leaves, with the lesson's settings, by 5-fold CV (not the test set)
for n_leaves in [7, 15, 31, 63]:
    m = LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=n_leaves, max_depth=-1, min_child_samples=10,
                       random_state=42, verbosity=-1)
    p = cross_val_predict(m, train_val[features], y_cls, cv=cv5)
    grown = m.fit(train_val[features], y_cls).booster_.trees_to_dataframe()
    most = grown[grown["split_feature"].isna()].groupby("tree_index").size().max()
    print(f"num_leaves={n_leaves:<3} most leaves actually grown {most:2d} | CV accuracy {np.mean(p == y_cls):.3f}"
          f" | F1 (pass) {f1_score(y_cls, p):.3f} | recall (fail) {recall_score(y_cls, p, pos_label=0):.3f}")

num_leaves=7   most leaves actually grown  7 | CV accuracy 0.756 | F1 (pass) 0.836 | recall (fail) 0.459


num_leaves=15  most leaves actually grown 13 | CV accuracy 0.764 | F1 (pass) 0.839 | recall (fail) 0.514


num_leaves=31  most leaves actually grown 13 | CV accuracy 0.764 | F1 (pass) 0.839 | recall (fail) 0.514


num_leaves=63  most leaves actually grown 13 | CV accuracy 0.764 | F1 (pass) 0.839 | recall (fail) 0.514


In [ ]:
import time

# XGBoost vs LightGBM on the same folds, each with the settings from its section
pair = {"XGBoost (alpha 1, lambda 5)": XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, reg_alpha=1, reg_lambda=5,
                                                     eval_metric="logloss", random_state=42),
        "LightGBM (the lesson's model)": LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=15, max_depth=5,
                                                       min_child_samples=10, random_state=42, verbosity=-1)}
for name, m in pair.items():
    start = time.time()
    p = cross_val_predict(m, train_val[features], y_cls, cv=cv5)
    print(f"{name:<30} CV accuracy {np.mean(p == y_cls):.3f} | F1 (pass) {f1_score(y_cls, p):.3f}"
          f" | recall (fail) {recall_score(y_cls, p, pos_label=0):.3f} | 5 fits in {time.time() - start:.1f}s")

XGBoost (alpha 1, lambda 5)    CV accuracy 0.795 | F1 (pass) 0.865 | recall (fail) 0.486 | 5 fits in 0.1s


LightGBM (the lesson's model)  CV accuracy 0.787 | F1 (pass) 0.859 | recall (fail) 0.486 | 5 fits in 0.1s


## 5. XGBoost vs LightGBM: comparison and best practices

In [ ]:
from sklearn.metrics import accuracy_score, precision_score

# Section 5: benchmark both on the same data. The lesson's two models, judged out-of-fold on the same 5 folds of the 127
models = {"XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8,
                                   eval_metric="logloss", random_state=42),
          "LightGBM": LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=15, max_depth=5, min_child_samples=10,
                                     random_state=42, verbosity=-1)}
results = []
for name, model in models.items():
    start = time.time()
    p = cross_val_predict(model, train_val[features], y_cls, cv=cv5)
    results.append({"Model": name, "Accuracy": accuracy_score(y_cls, p), "Precision (pass)": precision_score(y_cls, p),
                    "Recall (pass)": recall_score(y_cls, p), "F1 (pass)": f1_score(y_cls, p),
                    "Recall (fail)": recall_score(y_cls, p, pos_label=0), "Time (s)": time.time() - start})
print(pd.DataFrame(results).round(3).to_string(index=False))

   Model  Accuracy  Precision (pass)  Recall (pass)  F1 (pass)  Recall (fail)  Time (s)
 XGBoost     0.764             0.800          0.889      0.842          0.459     0.110
LightGBM     0.787             0.812          0.911      0.859          0.486     0.102


In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

# Stability across different data splits: 5-fold CV repeated 10 times with different shuffles (50 scores each)
rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
for name, model in models.items():
    acc = cross_val_score(model, train_val[features], y_cls, cv=rep)
    print(f"{name:<9} accuracy over 50 splits: mean {acc.mean():.3f} | std {acc.std():.3f} | range {acc.min():.2f}-{acc.max():.2f}")

XGBoost   accuracy over 50 splits: mean 0.763 | std 0.072 | range 0.60-0.88


LightGBM  accuracy over 50 splits: mean 0.766 | std 0.061 | range 0.64-0.88


In [ ]:
# Class imbalance: 37 failing vs 90 passing. Make FAIL the positive class (1) and weight it, then compare by CV
y_fail = 1 - y_cls                                            # 1 = fail (the minority), 0 = pass
weight = (y_fail == 0).sum() / (y_fail == 1).sum()
print(f"Fail {int(y_fail.sum())} of {len(y_fail)} ({y_fail.mean():.0%}); scale_pos_weight = {(y_fail == 0).sum()} / {(y_fail == 1).sum()} = {weight:.2f}")
variants = {"XGBoost": models["XGBoost"], "XGBoost, scale_pos_weight": XGBClassifier(**{**models["XGBoost"].get_params(), "scale_pos_weight": weight}),
            "LightGBM": models["LightGBM"], "LightGBM, class_weight=balanced": LGBMClassifier(**{**models["LightGBM"].get_params(), "class_weight": "balanced"})}
cv_fail_f1 = {}
for name, model in variants.items():
    p = cross_val_predict(model, train_val[features], y_fail, cv=cv5)
    cv_fail_f1[name] = f1_score(y_fail, p)
    print(f"{name:<32} recall (fail) {recall_score(y_fail, p):.3f} | precision (fail) {precision_score(y_fail, p):.3f}"
          f" | F1 (fail) {cv_fail_f1[name]:.3f} | accuracy {accuracy_score(y_fail, p):.3f}")

Fail 37 of 127 (29%); scale_pos_weight = 90 / 37 = 2.43
XGBoost                          recall (fail) 0.459 | precision (fail) 0.630 | F1 (fail) 0.531 | accuracy 0.764
XGBoost, scale_pos_weight        recall (fail) 0.568 | precision (fail) 0.488 | F1 (fail) 0.525 | accuracy 0.701


LightGBM                         recall (fail) 0.486 | precision (fail) 0.692 | F1 (fail) 0.571 | accuracy 0.787
LightGBM, class_weight=balanced  recall (fail) 0.622 | precision (fail) 0.575 | F1 (fail) 0.597 | accuracy 0.756


In [ ]:
from sklearn.metrics import confusion_matrix

# The rule, fixed before testing: for each framework, keep the variant with the higher CV F1 for fail. Then test ONCE
chosen = {fw: max((n for n in variants if n.startswith(fw)), key=cv_fail_f1.get) for fw in ["XGBoost", "LightGBM"]}
X_te, y_te_fail = test[features], 1 - test["Passed"].astype(int)
print(f"{len(y_te_fail)} test students, {int(y_te_fail.sum())} failing")
for fw, name in chosen.items():
    p = variants[name].fit(train_val[features], y_fail).predict(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te_fail, p).ravel()
    print(f"{name:<32} caught {tp} of {tp + fn} failing | false alarms {fp} | recall (fail) {recall_score(y_te_fail, p):.3f}"
          f" | precision (fail) {precision_score(y_te_fail, p, zero_division=0):.3f} | accuracy {accuracy_score(y_te_fail, p):.3f}")

23 test students, 7 failing
XGBoost                          caught 4 of 7 failing | false alarms 0 | recall (fail) 0.571 | precision (fail) 1.000 | accuracy 0.870
LightGBM, class_weight=balanced  caught 4 of 7 failing | false alarms 2 | recall (fail) 0.571 | precision (fail) 0.667 | accuracy 0.783


## 6. Mini project: XGBoost vs LightGBM

In [ ]:
# Section 6 mini project. Step 1: prepare the data, with the project's own 70 / 15 / 15 split. 1 = FAIL (the class to find)
df = pd.read_csv("student_data_150.csv")
print(f"{len(df)} students | missing {df[features].isna().sum().sum()} | duplicate rows {df.duplicated().sum()} | fail {(~df['Passed']).sum()}")
X, y = df[features], (~df["Passed"]).astype(int)
assert y.notna().all(), "Unexpected target labels"
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
for name, part in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name:<10} {len(part):3d} students, {part.sum():2d} failing")

150 students | missing 0 | duplicate rows 0 | fail 44
train      105 students, 31 failing
validation  22 students,  6 failing
test        23 students,  7 failing


In [ ]:
from sklearn.metrics import roc_auc_score

# Steps 2-3: baselines, trained on the training set and judged on the VALIDATION set (the test set stays untouched)
def judge(model, Xf, yf, Xe, ye):
    start = time.time(); model.fit(Xf, yf); seconds = time.time() - start
    p, proba = model.predict(Xe), model.predict_proba(Xe)[:, 1]
    return {"accuracy": accuracy_score(ye, p), "precision": precision_score(ye, p, zero_division=0), "recall": recall_score(ye, p),
            "F1": f1_score(ye, p), "ROC-AUC": roc_auc_score(ye, proba), "fit s": seconds}
base = {"XGBoost": dict(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8),
        "LightGBM": dict(n_estimators=100, learning_rate=0.05, num_leaves=15, max_depth=5, min_child_samples=10)}
make = {"XGBoost": lambda **k: XGBClassifier(eval_metric="logloss", random_state=42, **k),
        "LightGBM": lambda **k: LGBMClassifier(random_state=42, verbosity=-1, **k)}
rows = [{"Model": fw, **judge(make[fw](**base[fw]), X_train, y_train, X_val, y_val)} for fw in base]
print("Validation, fail as positive:")
print(pd.DataFrame(rows).drop(columns="fit s").round(3).to_string(index=False))

Validation, fail as positive:
   Model  accuracy  precision  recall    F1  ROC-AUC
 XGBoost     0.682      0.333   0.167 0.222    0.604
LightGBM     0.545      0.167   0.167 0.167    0.646


In [ ]:
# Step 4: experiments E1-E4 for both frameworks, on validation. LightGBM also varies min_child_samples
grid = {"E1": ({"max_depth": 3, "learning_rate": 0.01}, {"num_leaves": 7, "learning_rate": 0.01}),
        "E2": ({"max_depth": 3, "learning_rate": 0.05}, {"num_leaves": 15, "learning_rate": 0.05}),
        "E3": ({"max_depth": 5, "learning_rate": 0.1}, {"num_leaves": 31, "learning_rate": 0.1}),
        "E4": ({"max_depth": 7, "learning_rate": 0.1}, {"num_leaves": 63, "learning_rate": 0.1})}
results = []
for e, (xp, lp) in grid.items():
    results.append({"Exp": e, "Model": "XGBoost", "Setting": str(xp), **judge(make["XGBoost"](n_estimators=100, **xp), X_train, y_train, X_val, y_val)})
    for mcs in [5, 20]:
        results.append({"Exp": e, "Model": "LightGBM", "Setting": str({**lp, "min_child_samples": mcs}),
                        **judge(make["LightGBM"](n_estimators=100, min_child_samples=mcs, **lp), X_train, y_train, X_val, y_val)})
experiments = pd.DataFrame(results)
print(experiments[["Exp", "Model", "Setting", "recall", "F1", "ROC-AUC"]].round(3).to_string(index=False))

Exp    Model                                                            Setting  recall    F1  ROC-AUC
 E1  XGBoost                            {'max_depth': 3, 'learning_rate': 0.01}   0.000 0.000    0.521
 E1 LightGBM   {'num_leaves': 7, 'learning_rate': 0.01, 'min_child_samples': 5}   0.000 0.000    0.589
 E1 LightGBM  {'num_leaves': 7, 'learning_rate': 0.01, 'min_child_samples': 20}   0.000 0.000    0.490
 E2  XGBoost                            {'max_depth': 3, 'learning_rate': 0.05}   0.167 0.250    0.615
 E2 LightGBM  {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 5}   0.333 0.308    0.656
 E2 LightGBM {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 20}   0.333 0.308    0.552
 E3  XGBoost                             {'max_depth': 5, 'learning_rate': 0.1}   0.333 0.364    0.688
 E3 LightGBM   {'num_leaves': 31, 'learning_rate': 0.1, 'min_child_samples': 5}   0.333 0.308    0.562
 E3 LightGBM  {'num_leaves': 31, 'learning_rate': 0.1, 'min_child_samples

In [ ]:
# The rule, fixed before testing: for each framework, the setting with the best validation F1 for fail
# (ties: higher ROC-AUC, then the simpler, earlier setting)
chosen = {}
for fw in ["XGBoost", "LightGBM"]:
    best = experiments[experiments["Model"] == fw].sort_values(["F1", "ROC-AUC"], ascending=False, kind="stable").iloc[0]
    chosen[fw] = best
    print(f"{fw:<9} chosen {best['Exp']} {best['Setting']} | validation F1 {best['F1']:.3f} | recall {best['recall']:.3f} | ROC-AUC {best['ROC-AUC']:.3f}")

XGBoost   chosen E3 {'max_depth': 5, 'learning_rate': 0.1} | validation F1 0.364 | recall 0.333 | ROC-AUC 0.688
LightGBM  chosen E2 {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 5} | validation F1 0.308 | recall 0.333 | ROC-AUC 0.656


In [ ]:
import ast

# Step 5: retrain the chosen settings on train + validation (127 students), then evaluate ONCE on the 23 test students
X_tv, y_tv2 = pd.concat([X_train, X_val]), pd.concat([y_train, y_val])
final_rows = []
for fw, best in chosen.items():
    params = ast.literal_eval(best["Setting"])
    m = make[fw](n_estimators=100, **params)
    r = judge(m, X_tv, y_tv2, X_test, y_test)
    tn, fp, fn, tp = confusion_matrix(y_test, m.predict(X_test)).ravel()
    final_rows.append({"Model": fw, **r})
    print(f"{fw:<9} caught {tp} of {tp + fn} failing | false alarms {fp} | missed {fn} | correctly left alone {tn}")
print(pd.DataFrame(final_rows).round(3).to_string(index=False))

XGBoost   caught 4 of 7 failing | false alarms 5 | missed 3 | correctly left alone 11
LightGBM  caught 4 of 7 failing | false alarms 5 | missed 3 | correctly left alone 11
   Model  accuracy  precision  recall  F1  ROC-AUC  fit s
 XGBoost     0.652      0.444   0.571 0.5    0.732  0.018
LightGBM     0.652      0.444   0.571 0.5    0.750  0.018
